In [ ]:
"""
eurosat_vit_app.py

EuroSAT Land Use Classification with Vision Transformer (single-model Gradio app)

"""

import os
import traceback
from io import BytesIO
from typing import Tuple, Optional, Dict

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import requests
import gradio as gr

# ========================================================================
# SECTION 1: Setup and Configuration
# ========================================================================

# Mount Google Drive (for Colab)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
    DRIVE_MOUNTED = True
    DRIVE_PREFIX = '/content/drive/MyDrive/'
except Exception:
    print("ℹ️ Not running in Colab - using local file system")
    DRIVE_MOUNTED = False
    DRIVE_PREFIX = './'

# EuroSAT class names
EUROSAT_CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

# ========================================================================
# SECTION 2: Vision Transformer Architecture (FIXED)
# ========================================================================

class PatchEmbedding(nn.Module):
    """Convert image to patch embeddings"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        # FIXED: Changed 'proj' to 'projection' to match trained model
        self.projection = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

        # initialize
        nn.init.xavier_uniform_(self.projection.weight)
        if self.projection.bias is not None:
            nn.init.zeros_(self.projection.bias)

    def forward(self, x):
        # x: (B, C, H, W) -> (B, n_patches, embed_dim)
        x = self.projection(x)              # (B, E, H/ps, W/ps)
        x = x.flatten(2).transpose(1, 2)    # (B, N, E)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim=768, num_heads=12, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        nn.init.xavier_uniform_(self.qkv.weight)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x

class MLP(nn.Module):
    def __init__(self, embed_dim=768, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=768, num_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.mlp = MLP(embed_dim, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class EuroSATViT(nn.Module):
    """Vision Transformer tailored for EuroSAT small-scale use"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768,
                 depth=8, num_heads=12, mlp_ratio=4.0, num_classes=10, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        n_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout)

        # transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout) for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.head = nn.Linear(embed_dim, num_classes)

        # init
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                       # (B, N, E)
        cls_tokens = self.cls_token.expand(B, -1, -1) # (B, 1, E)
        x = torch.cat([cls_tokens, x], dim=1)         # (B, N+1, E)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        cls_final = x[:, 0]   # (B, E)
        return self.head(cls_final)

# ========================================================================
# SECTION 3: Image Preprocessing
# ========================================================================

def get_transforms():
    """Image transforms for ViT (224x224)"""
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

transform = get_transforms()

# ========================================================================
# SECTION 4: Model Loading (robust + cache + compatibility)
# ========================================================================

_MODEL_CACHE: Dict[str, Tuple[nn.Module, dict]] = {}

def _resolve_path(path: str) -> str:
    if not path:
        return path
    path = os.path.expanduser(path)
    if DRIVE_MOUNTED and not path.startswith('/content/drive/') and not os.path.isabs(path):
        # treat relative paths in Colab as relative to Drive root
        path = os.path.join(DRIVE_PREFIX, path)
    return path

def load_vit_checkpoint(model_path: str) -> Tuple[Optional[nn.Module], dict]:
    """
    Robust loader for ViT checkpoints. Handles:
    - {'model_state_dict': ...}
    - {'state_dict': ...}
    - direct state_dict
    - keys prefixed with 'module.'
    - architecture naming mismatches (proj vs projection)
    - caches loaded model for reuse
    """
    try:
        model_path = _resolve_path(model_path or "")
        if not model_path or not os.path.exists(model_path):
            return None, {"error": f"Model file not found: {model_path}"}

        abs_path = os.path.abspath(model_path)
        if abs_path in _MODEL_CACHE:
            model, info = _MODEL_CACHE[abs_path]
            info_copy = dict(info)
            info_copy["cached"] = True
            return model, info_copy

        # Use consistent device mapping
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)

        # instantiate model structure (match the training config used earlier)
        model = EuroSATViT(
            img_size=224,
            patch_size=16,
            in_chans=3,
            embed_dim=768,
            depth=8,
            num_heads=12,
            mlp_ratio=4.0,
            num_classes=len(EUROSAT_CLASSES),
            dropout=0.1
        )

        # get state_dict from wrapper varieties
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                # Possibly direct state_dict-like dict
                tensor_mask = [isinstance(v, torch.Tensor) for v in checkpoint.values()]
                if all(tensor_mask) and len(tensor_mask) > 0:
                    state_dict = checkpoint
                else:
                    # fallback: pick tensor-like items
                    state_dict = {k: v for k, v in checkpoint.items() if isinstance(v, torch.Tensor)}
        else:
            state_dict = checkpoint

        # strip "module." prefix if present (DataParallel)
        new_state = {}
        for k, v in state_dict.items():
            new_key = k[len('module.'):] if k.startswith('module.') else k
            new_state[new_key] = v

        # Handle naming mismatches (proj <-> projection)
        fixed_state = {}
        for k, v in new_state.items():

            if 'patch_embed.proj.' in k:
                new_k = k.replace('patch_embed.proj.', 'patch_embed.projection.')
                fixed_state[new_k] = v
                print(f"🔧 Renamed key: {k} -> {new_k}")
            else:
                fixed_state[k] = v

        # try loading with strict=False for tolerance
        load_res = model.load_state_dict(fixed_state, strict=False)

        # move to device and eval
        model.to(device)
        model.eval()

        total_params = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

        missing = getattr(load_res, 'missing_keys', None) or []
        unexpected = getattr(load_res, 'unexpected_keys', None) or []

        # Check for critical issues
        critical_missing = [k for k in missing if not k.endswith('.bias')]
        if critical_missing:
            print(f"⚠️ WARNING: Critical keys still missing: {critical_missing[:5]}")

        info = {
            "path": abs_path,
            "total_params": total_params,
            "trainable_params": trainable,
            "accuracy": checkpoint.get('val_accuracy', checkpoint.get('test_acc', checkpoint.get('accuracy', None))) if isinstance(checkpoint, dict) else None,
            "epoch": checkpoint.get('epoch', None) if isinstance(checkpoint, dict) else None,
            "missing_keys": missing,
            "unexpected_keys": unexpected
        }

        _MODEL_CACHE[abs_path] = (model, info)
        return model, dict(info)

    except Exception as e:
        tb = traceback.format_exc()
        return None, {"error": str(e), "traceback": tb}

# ========================================================================
# SECTION 5: Prediction Function
# ========================================================================

def predict_vit(model: nn.Module, image: Image.Image):
    """Run inference and return top-5 predictions and full probability vector"""
    x = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.nn.functional.softmax(logits, dim=1)[0].cpu().numpy()

    predicted_idx = int(np.argmax(probs))
    predicted_class = EUROSAT_CLASSES[predicted_idx]
    confidence = float(probs[predicted_idx])

    top5_idx = np.argsort(probs)[-5:][::-1]
    top5 = [(EUROSAT_CLASSES[i], float(probs[i])) for i in top5_idx]

    return {
        "predicted_class": predicted_class,
        "confidence": confidence,
        "all_probabilities": probs,
        "top5_classes": [c for c, p in top5],
        "top5_probs": [p for c, p in top5],
        "top5": top5
    }

# ========================================================================
# SECTION 6: Visualization (matplotlib horizontal bar)
# ========================================================================

def create_bar_chart(results: dict):
    """Create horizontal bar chart for top-5 predictions (matplotlib)"""
    if not results or 'top5' not in results:
        return None

    top5 = results['top5']
    classes = [c for c, p in top5]
    probs = [p * 100 for c, p in top5]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    y_pos = np.arange(len(classes))
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(classes))]

    bars = ax.barh(y_pos, probs, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(classes)
    ax.invert_yaxis()
    ax.set_xlim(0, max(100, max(probs) + 5))
    ax.set_xlabel('Confidence (%)')
    ax.set_title('Top-5 Predictions')

    for bar, p in zip(bars, probs):
        ax.text(p + 1, bar.get_y() + bar.get_height() / 2, f"{p:.1f}%", va='center', fontweight='bold')

    plt.tight_layout()
    return fig

# ========================================================================
# SECTION 7: Helpers (image loading)
# ========================================================================

def load_image_from_url(url: str) -> Tuple[Optional[Image.Image], Optional[str]]:
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        return image, None
    except Exception as e:
        return None, str(e)

# ========================================================================
# SECTION 8: Gradio Interface
# ========================================================================

_loaded_model: Optional[nn.Module] = None
_loaded_model_info: Optional[dict] = None

def load_model_gradio(model_path: str):
    """Load ViT model for Gradio UI"""
    global _loaded_model, _loaded_model_info
    if not model_path or not model_path.strip():
        return "⚠️ Please provide a model path (.pth)"

    model, info = load_vit_checkpoint(model_path.strip())
    if model is None:
        err = info.get("error", "Unknown error")
        tb = info.get("traceback", "")
        return f"❌ Failed to load model: {err}\n\n{tb[:1000]}"

    _loaded_model = model
    _loaded_model_info = info

    lines = ["✅ Vision Transformer loaded successfully!"]
    if info.get("accuracy") is not None:
        lines.append(f"• Accuracy (from checkpoint): {info['accuracy']}")
    if info.get("epoch") is not None:
        lines.append(f"• Epoch: {info['epoch']}")
    lines.append(f"• Parameters: {info['total_params']:,} (trainable: {info['trainable_params']:,})")

    missing = info.get("missing_keys", []) or []
    unexpected = info.get("unexpected_keys", []) or []

    if missing or unexpected:
        lines.append(f"• Keys status: missing={len(missing)}, unexpected={len(unexpected)}")
        if len(missing) > 0:
            lines.append(f"  - Missing keys (first 3): {missing[:3]}")
        if len(unexpected) > 0:
            lines.append(f"  - Unexpected keys (first 3): {unexpected[:3]}")
    else:
        lines.append("• ✅ All keys matched perfectly!")

    if info.get("cached"):
        lines.append("• Loaded from cache")

    return "\n".join(lines)

def load_url_to_image_gradio(url: str):
    """Helper to load URL into Gradio image input"""
    if not url or not url.strip():
        return None
    img, err = load_image_from_url(url.strip())
    if err:
        return None
    return img

def predict_gradio(image):
    """Run inference from Gradio UI"""
    global _loaded_model, _loaded_model_info
    if _loaded_model is None:
        return None, "❌ Please load a model first.", None

    if image is None:
        return None, "⚠️ Please upload an image or provide a URL.", None

    try:
        results = predict_vit(_loaded_model, image)
    except Exception as e:
        tb = traceback.format_exc()
        return None, f"❌ Inference error: {e}\n\n{tb[:1000]}", None

    # Compose textual result
    results_text = f"""✅ Prediction Complete!

🎯 Predicted Class: {results['predicted_class']}
📈 Confidence: {results['confidence'] * 100:.2f}%

📊 Top 5 Predictions:
"""
    for cls, p in zip(results['top5_classes'], results['top5_probs']):
        results_text += f"• {cls}: {p*100:.1f}%\n"

    # Create chart
    chart = create_bar_chart(results)

    return image, results_text, chart

# Build Gradio interface
def create_interface():
    with gr.Blocks(theme=gr.themes.Soft(), title="EuroSAT Vision Transformer Classifier") as demo:
        gr.Markdown("""
        # 🛰️ EuroSAT Land Use Classification — Vision Transformer (ViT)

        Upload a satellite or aerial image (or provide a URL) to classify into one of 10 land use classes.
        """)

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 🔧 Step 1: Load ViT Model")
                model_path_input = gr.Textbox(
                    label="ViT Model Path (.pth)",
                    placeholder="/content/drive/MyDrive/weights/final_eurosat_vit_model.pth",
                    value="/content/drive/MyDrive/weights/final_eurosat_vit_model.pth",
                    info="Path to Vision Transformer weights (.pth)"
                )
                load_btn = gr.Button("📥 Load Model", variant="primary")
                model_status = gr.Textbox(label="Model Status", lines=10, interactive=False)

                gr.Markdown("""
                ### 📝 Path Examples:
                - Google Drive (Colab): `/content/drive/MyDrive/weights/vit_eurosat_best.pth`
                - Relative: `weights/vit_eurosat_best.pth`
                - Local: `/path/to/vit_eurosat_best.pth`
                """)

            with gr.Column(scale=1):
                gr.Markdown("### 🖼️ Step 2: Upload Image / URL")
                image_input = gr.Image(label="Upload Image (optional)", type="pil", height=300)
                gr.Markdown("**Or paste an image URL:**")
                image_url = gr.Textbox(label="Image URL", placeholder="https://example.com/satellite-image.jpg")
                load_url_btn = gr.Button("🔗 Load from URL")
                predict_btn = gr.Button("🔍 Classify Image", variant="primary", size="lg")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📊 Results")
                results_text = gr.Textbox(label="Prediction Results", lines=12, interactive=False)

            with gr.Column(scale=1):
                gr.Markdown("### 📈 Confidence Chart")
                results_chart = gr.Plot(label="Top-5 Predictions")

        gr.Markdown("""
        ---
        ### 📖 Instructions:
        1. Enter the path to your ViT checkpoint (.pth) and click **Load Model**.
        2. Upload an image or paste an image URL and click **Load from URL**.
        3. Click **Classify Image** to run inference and view results.

        ### ✅ Expected Status After Loading:
        - **missing_keys=0, unexpected_keys=0** means the model loaded correctly
        - If you see mismatches, the predictions may be inaccurate
        """)

        # Events
        load_btn.click(fn=load_model_gradio, inputs=[model_path_input], outputs=[model_status])
        load_url_btn.click(fn=load_url_to_image_gradio, inputs=[image_url], outputs=[image_input])
        predict_btn.click(fn=predict_gradio, inputs=[image_input], outputs=[image_input, results_text, results_chart])

    return demo

# ========================================================================
# SECTION 9: Launch
# ========================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("🛰️ EuroSAT LAND USE CLASSIFICATION - Vision Transformer (ViT)")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Google Drive: {'✅ Mounted' if DRIVE_MOUNTED else '❌ Not mounted'}")
    print(f"Classes: {len(EUROSAT_CLASSES)}")
    print("=" * 70)

    demo = create_interface()
    demo.launch(server_name="0.0.0.0", server_port=7860, share=False, debug=True)

Mounted at /content/drive
✅ Google Drive mounted successfully!
🔧 Using device: cuda
🛰️ EuroSAT LAND USE CLASSIFICATION - Vision Transformer (ViT)
Device: cuda
Google Drive: ✅ Mounted
Classes: 10
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>